# Clusters

# Map

In [20]:
import geopandas as gpd
import polars as pl
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pandas as pd

ids = pl.read_csv("/root/2025-Project-188/data/train_file_ids.csv")["file_id"].to_list()
df = pd.read_csv('/root/2025-Project-188/data/river_geo_vector.csv', index_col="gauge_id")
df = df.loc[ids]
X = df.copy()

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

k_optimal = 9
kmeans = KMeans(n_clusters=k_optimal, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df['cluster'] = clusters

coords = df[['lon', 'lat']].to_numpy()

min_lon, min_lat, max_lon, max_lat = df['lon'].min(), df['lat'].min(), df['lon'].max(), df['lat'].max(),
buffer = 1

# 4. Create figure & add Natural Earth features
fig = plt.figure(figsize=(20, 18))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND.with_scale("10m"))
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.5)
ax.set_extent([min_lon - buffer, max_lon + buffer,
               min_lat - buffer, max_lat + buffer])

# 5. Plot each cluster in a distinct color
unique_clusters = np.sort(df['cluster'].unique())
cmap = get_cmap('tab10', len(unique_clusters))
for idx, cluster in enumerate(unique_clusters):
    subset = df[df['cluster'] == cluster]
    ax.scatter(
        subset['lon'],
        subset['lat'],
        s=100,
        color=cmap(idx),
        edgecolor='black',
        linewidth=0.5,
        alpha=0.8,
        transform=ccrs.PlateCarree(),
        zorder=5,
        label=f'Cluster {cluster}'
    )

ax.legend(title='Clusters', loc='upper right')
ax.set_title("Training River Gauge Locations — Clusters", pad=12)

# 6. Save as PDF (vector format)
plt.savefig(
    "/root/2025-Project-188/artifacts/plots/train_gauges_cartopy_10m_clusters.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.close()


/tmp/ipykernel_26739/222976773.py:46: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = get_cmap('tab10', len(unique_clusters))


In [21]:
df.to_csv("/root/2025-Project-188/artifacts/dataset/clusters.csv")

In [23]:
df.dtypes

for_pc_sse    float64
crp_pc_sse    float64
inu_pc_ult    float64
ire_pc_sse    float64
lka_pc_use    float64
prm_pc_sse    float64
pst_pc_sse    float64
cly_pc_sav    float64
slt_pc_sav    float64
snd_pc_sav    float64
kar_pc_sse    float64
urb_pc_sse    float64
gwt_cm_sav    float64
lkv_mc_usu    float64
rev_mc_usu    float64
sgr_dk_sav    float64
slp_dg_sav    float64
ws_area       float64
ele_mt_sav    float64
height_bs     float64
lat           float64
lon           float64
cluster         int32
dtype: object

In [29]:
import pandas as pd
import numpy as np

# df : DataFrame with numeric features + a column 'cluster'

# 1. Centroids (already in your `cluster_profile`)
centroids = (
    df.groupby('cluster')
      .mean(numeric_only=True)
)

# 2. Overall mean & std
overall_mean = df.mean(numeric_only=True)
overall_std  = df.std(numeric_only=True)

# 3. z-score distances
z_dist = (centroids - overall_mean) / overall_std
z_dist = z_dist.abs()                 # absolute distances
z_dist.columns = [f'{c}_z' for c in z_dist.columns]

# 4. For each cluster, pick the top-k features
topk = 3
top_feats = (
    z_dist
    .apply(lambda row: row.nlargest(topk).index.tolist(), axis=1)
    .to_frame(name='top_features')
)

display(top_feats)

,top_features
cluster,
0,"[lat_z, for_pc_sse_z, gwt_cm_sav_z]"
1,"[crp_pc_sse_z, for_pc_sse_z, pst_pc_sse_z]"
2,"[cly_pc_sav_z, slt_pc_sav_z, snd_pc_sav_z]"
3,"[inu_pc_ult_z, lka_pc_use_z, lat_z]"
4,"[sgr_dk_sav_z, gwt_cm_sav_z, slp_dg_sav_z]"
5,"[lon_z, lat_z, slp_dg_sav_z]"
6,"[prm_pc_sse_z, lon_z, ws_area_z]"
7,"[ele_mt_sav_z, height_bs_z, prm_pc_sse_z]"
8,"[lkv_mc_usu_z, cly_pc_sav_z, slt_pc_sav_z]"


In [30]:
topk = 3                                 # number of ranks you kept earlier
rank_cols = [f'rank_{i+1}' for i in range(topk)]

top_feats_wide = (
    top_feats['top_features']            # take the list column
      .apply(pd.Series, index=rank_cols) # split list → Series
      .join(top_feats.drop(columns='top_features'))
      .reset_index()                     # optional: make cluster a normal column
)

print(top_feats_wide)

   cluster        rank_1        rank_2        rank_3
0        0         lat_z  for_pc_sse_z  gwt_cm_sav_z
1        1  crp_pc_sse_z  for_pc_sse_z  pst_pc_sse_z
2        2  cly_pc_sav_z  slt_pc_sav_z  snd_pc_sav_z
3        3  inu_pc_ult_z  lka_pc_use_z         lat_z
4        4  sgr_dk_sav_z  gwt_cm_sav_z  slp_dg_sav_z
5        5         lon_z         lat_z  slp_dg_sav_z
6        6  prm_pc_sse_z         lon_z     ws_area_z
7        7  ele_mt_sav_z   height_bs_z  prm_pc_sse_z
8        8  lkv_mc_usu_z  cly_pc_sav_z  slt_pc_sav_z
